# LLM Server for Medical Diagnosis - Colab Setup

This notebook runs a medical LLM server in Google Colab that integrates with your main Flask application.

**Flow:**
1. Colab notebook runs Flask server with LLM model
2. ngrok exposes the server publicly
3. Your main app.py connects to this public URL
4. Chat requests are routed from app.py → Colab LLM server → responses back to frontend

## Step 1: Install Required Dependencies

In [ ]:
# Install dependencies for Colab
!pip install -q transformers torch accelerate bitsandbytes peft flask pyngrok --upgrade

## Step 2: Setup and Load the LLM Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Choose your model - options:
# - "mistralai/Mistral-7B-Instruct-v0.2" (recommended, ~7GB)
# - "microsoft/Phi-3-mini-4k-instruct" (smaller, ~3GB)
# - "HuggingFaceH4/zephyr-7b-beta" (good for medical context)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

logger.info("=" * 60)
logger.info("Loading LLM Model for Medical Diagnosis")
logger.info("=" * 60)

logger.info(f"Model: {MODEL_NAME}")
logger.info("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
logger.info("✓ Tokenizer loaded")

logger.info("Setting up 4-bit quantization for memory efficiency...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
logger.info("✓ Quantization configured")

logger.info("Loading model (this may take 2-3 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
logger.info("✓ Model loaded")

logger.info("Creating text generation pipeline...")
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)
logger.info("✓ Pipeline ready!")
logger.info("=" * 60)

## Step 3: Create Flask Server with LLM Endpoints

In [ ]:
from flask import Flask, request, jsonify, Response, stream_with_context
import threading
import json
from transformers import TextIteratorStreamer

app = Flask(__name__)

def generate_response(prompt: str, max_tokens: int = 200, temperature: float = 0.7):
    """Generate response using the loaded model"""
    try:
        output = pipe(
            prompt,
            max_new_tokens=max_tokens,
            max_length=None,
            pad_token_id=tokenizer.eos_token_id,
            temperature=temperature,
            do_sample=True,
            top_p=0.95,
            return_full_text=False
        )
        return output[0]["generated_text"]
    except Exception as e:
        logger.error(f"Error generating response: {e}")
        return f"Error: {str(e)}"


@app.route("/", methods=["GET"])
def home():
    """Health check endpoint"""
    return jsonify({
        "status": "ready",
        "message": "LLM Server is running!",
        "model": MODEL_NAME
    })


@app.route("/health", methods=["GET"])
def health():
    """Extended health check"""
    return jsonify({
        "status": "healthy",
        "model_loaded": True,
        "model_name": MODEL_NAME,
        "gpu_available": torch.cuda.is_available()
    })


@app.route("/chat", methods=["POST"])
def chat():
    """
    Chat endpoint - accepts prompt and generates response
    
    Expected JSON:
    {
        "prompt": "What are symptoms of diabetes?",
        "max_tokens": 200,
        "temperature": 0.7
    }
    """
    try:
        data = request.get_json()
        
        if not data or "prompt" not in data:
            return jsonify({"error": "No prompt provided"}), 400
        
        prompt = data.get("prompt", "")
        max_tokens = data.get("max_tokens", 200)
        temperature = data.get("temperature", 0.7)
        
        if not prompt.strip():
            return jsonify({"error": "Prompt cannot be empty"}), 400
        
        logger.info(f"Processing prompt: {prompt[:50]}...")
        response = generate_response(prompt, max_tokens, temperature)
        
        return jsonify({
            "response": response,
            "status": "success",
            "prompt": prompt,
            "model": MODEL_NAME
        })
    
    except Exception as e:
        logger.error(f"Chat error: {e}")
        return jsonify({
            "error": str(e),
            "status": "error"
        }), 500


@app.route("/chat/stream", methods=["POST"])
def chat_stream():
    """
    Streaming chat endpoint - streams tokens as they are generated using TextIteratorStreamer
    """
    def generate():
        try:
            data = request.get_json()
            
            if not data or "prompt" not in data:
                yield f"data: {json.dumps({'error': 'No prompt provided'})}\n\n"
                return
            
            prompt = data.get("prompt", "")
            max_tokens = data.get("max_tokens", 200)
            temperature = data.get("temperature", 0.7)
            
            if not prompt.strip():
                yield f"data: {json.dumps({'error': 'Prompt cannot be empty'})}\n\n"
                return
            
            logger.info(f"Streaming prompt: {prompt[:50]}...")
            
            # Setup actual streaming
            streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
            
            # Run generation in a background thread to allow yielding tokens
            generation_kwargs = dict(
                max_new_tokens=max_tokens,
                max_length=None,
                pad_token_id=tokenizer.eos_token_id,
                temperature=temperature,
                do_sample=True,
                streamer=streamer,
                return_full_text=False
            )
            
            thread = threading.Thread(target=pipe, args=(prompt,), kwargs=generation_kwargs)
            thread.start()
            
            # Yield tokens as they are generated (blocks until empty stream)
            for new_text in streamer:
                yield f"data: {json.dumps({'token': new_text})}\n\n"
            
            yield f"data: {json.dumps({'token': '[DONE]'})}\n\n"
        
        except Exception as e:
            logger.error(f"Stream error: {e}")
            yield f"data: {json.dumps({'error': str(e)})}\n\n"
            yield f"data: {json.dumps({'token': '[DONE]'})}\n\n"
            
    return Response(
        stream_with_context(generate()), 
        mimetype="text/event-stream",
        headers={
            'Cache-Control': 'no-cache',
            'Connection': 'keep-alive',
            'X-Accel-Buffering': 'no'
        }
    )

logger.info("Flask app configured and ready!")
logger.info("Routes available:")
logger.info("  GET  / - Health check")
logger.info("  GET  /health - Extended health check")
logger.info("  POST /chat - Send prompt, get response")
logger.info("  POST /chat/stream - Send prompt, stream tokens")

## Step 4: Expose Server with ngrok - Get Your Public URL

In [ ]:
import threading
import time
import os
import subprocess
import re

logger.info("Starting Flask server...")
logger.info("=" * 60)

# Run Flask app in background
def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

logger.info("Flask server started on port 5000")
time.sleep(2)

try:
    from google.colab.output import eval_js
    proxy_url = eval_js("google.colab.kernel.proxyPort(5000)")
    print("\n" + "=" * 70)
    print("🌐 COLAB PROXY URL")
    print("This URL only works in the same browser where you are logged into Colab:")
    print(proxy_url)
    print("=" * 70)
except ImportError:
    pass

print("\n" + "=" * 70)
print("🌐 SETTING UP CLOUDFLARE TUNNEL (Highly Reliable)...")

if not os.path.exists("cloudflared"):
    os.system("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    os.system("chmod +x cloudflared")

# Start Cloudflared tunnel
with open("cloudflared.log", "w") as log_file:
    tunnel_process = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5000'],
                                      stdout=log_file, stderr=log_file)
                                      
print("Waiting for Cloudflare URL (takes ~6-8 seconds)...")
time.sleep(8)

cf_url = None
with open("cloudflared.log", "r") as f:
    log_content = f.read()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_content)
    if match:
        cf_url = match.group(0)

if cf_url:
    print("\n" + "=" * 70)
    print("✅ SUCCESS! YOUR PUBLIC ENDPOINT:")
    print(cf_url)
    print("\nCopy that https://... URL and use it as your EXTERNAL_LLM_URL in app.py!")
    print("=" * 70)
else:
    print("\nCould not find Cloudflare URL in time. Output so far:")
    print(log_content)

## Step 5: Test the LLM Server

Test the endpoints locally to verify they're working:

In [ ]:
import time
import requests

# Give Flask a moment to start
time.sleep(2)

# Test health endpoint
logger.info("Testing health endpoint...")
try:
    response = requests.get("http://localhost:5000/health", timeout=10)
    logger.info(f"Health check: {response.json()}")
except Exception as e:
    logger.error(f"Health check failed: {e}")

# Test chat endpoint
logger.info("\nTesting chat endpoint...")
test_prompts = [
    "What are the first symptoms of diabetes?",
    "Explain blood test results",
    "How to diagnose anemia?"
]

for prompt in test_prompts:
    try:
        logger.info(f"\nPrompt: {prompt}")
        response = requests.post(
            "http://localhost:5000/chat",
            json={"prompt": prompt, "max_tokens": 150},
            timeout=30
        )
        if response.status_code == 200:
            result = response.json()
            logger.info(f"Response: {result['response'][:200]}...")
        else:
            logger.error(f"Server error: {response.status_code}")
    except Exception as e:
        logger.error(f"Request failed: {e}")

logger.info("\n✓ Server testing complete!")

## Step 6: Integration with Your Main App

Now update your main `app.py` to use this external LLM server.

### Configuration Instructions:

1. **Copy the public URL from the Serveo output above** (e.g., `https://xxxxx.serveo.net`)
2. **Update your app.py** - Set the external LLM URL in the ChatModel initialization or as an environment variable
3. **Restart your local Flask server** (`python app.py`)
4. **Test the integration** - Send chat requests through your web UI

The configuration is already added to your updated `app.py` - just uncomment and set the URL!

## Step 7: Keep Server Running

The cell below keeps the notebook running. Don't stop it while you're using the LLM from your main app.

In [ ]:
# Keep the server running
print("\n" + "=" * 70)
print("🎯 LLM SERVER READY FOR PRODUCTION")
print("=" * 70)
print("\nYour Colab LLM server is now running and accessible.")
print("\nStatus:")
print("  ✓ Model loaded and ready")
print("  ✓ Flask server running on port 5000")
print("  ✓ Serveo tunnel active (in previous cell)")
print("\nAvailable endpoints:")
print("  GET  <public-url>/          - Health check")
print("  GET  <public-url>/health    - Extended health")
print("  POST <public-url>/chat      - Chat endpoint")
print("  POST <public-url>/chat/stream - Streaming endpoint")
print("\n" + "=" * 70)
print("To stop the server: Click 'Interrupt execution' or close this notebook")
print("=" * 70)

# Keep notebook running
import time
try:
    while True:
        time.sleep(60)
        print(f"[{time.strftime('%H:%M:%S')}] Server still running...")
except KeyboardInterrupt:
    logger.info("Server shutdown requested")
    print("✓ Server stopped")